In [1]:
import cv2
from mtcnn import MTCNN

In [2]:
import numpy as np

In [3]:
from tensorflow.keras.models import load_model

In [4]:
model = load_model("VGG_final.keras")

In [6]:
import cv2
import numpy as np
from mtcnn import MTCNN
from tensorflow.keras.applications.vgg16 import preprocess_input
from collections import deque
from statistics import mode

cap = cv2.VideoCapture(0)
detector = MTCNN()

# ── Tunable parameters ─────────────────────────────────────────────────────────
HISTORY_LEN    = 15
EMA_ALPHA      = 0.15
JUMP_THRESHOLD = 12

# ── Padding (% of detected box size) ──────────────────────────────────────────
PAD_TOP    = 0.30
PAD_BOTTOM = 0.12
PAD_LEFT   = 0.12
PAD_RIGHT  = 0.12

smoothed_age        = None
age_history         = deque(maxlen=HISTORY_LEN)
gender_history_vote = deque(maxlen=15)   # 1 = Female, 0 = Male

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img_h, img_w = frame.shape[:2]
    output = detector.detect_faces(frame)

    for single_output in output:
        x, y, w, h = single_output['box']
        confidence  = single_output['confidence']

        # ── Padded & clamped crop coordinates ─────────────────────────
        pad_top    = int(h * PAD_TOP)
        pad_bottom = int(h * PAD_BOTTOM)
        pad_left   = int(w * PAD_LEFT)
        pad_right  = int(w * PAD_RIGHT)

        x1 = max(0,     x - pad_left)
        y1 = max(0,     y - pad_top)
        x2 = min(img_w, x + w + pad_right)
        y2 = min(img_h, y + h + pad_bottom)

        face = frame[y1:y2, x1:x2]
        if face.size == 0:
            continue

        # ── Preprocess ────────────────────────────────────────────────
        face_input = cv2.resize(face, (128, 128)).astype("float32")
        face_input = preprocess_input(face_input)
        face_input = np.expand_dims(face_input, axis=0)

        # ── Predict ───────────────────────────────────────────────────
        pred        = model.predict(face_input, verbose=0)
        current_age = pred[0][0][0] * 116
        gender      = pred[1][0][0]

        # ── Age smoothing ──────────────────────────────────────────────
        if smoothed_age is None:
            smoothed_age = current_age

        if abs(current_age - smoothed_age) < JUMP_THRESHOLD:
            smoothed_age = (1 - EMA_ALPHA) * smoothed_age + EMA_ALPHA * current_age
            age_history.append(current_age)
        else:
            age_history.append(current_age)

        display_age = int(np.median(age_history)) if age_history else int(smoothed_age)

        # ── Gender: vote then mode ─────────────────────────────────────
        gender_history_vote.append(1 if gender > 0.5 else 0)
        gender_label = "Female" if mode(gender_history_vote) == 1 else "Male"

        # ── Draw tight box (green) ─────────────────────────────────────
        tx, ty = max(0, x), max(0, y)
        cv2.rectangle(frame, (tx, ty), (tx+w, ty+h), (0, 255, 0), 2)

        # ── Draw padded box (orange) ───────────────────────────────────
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 165, 0), 1)

        # ── Draw facial landmarks ──────────────────────────────────────
        for point in single_output['keypoints'].values():
            cv2.circle(frame, point, 3, (0, 0, 
                                         255), -1)

        # ── Overlay label ──────────────────────────────────────────────
        label      = f"{gender_label}, {display_age}"
        conf_label = f"Det:{confidence:.2f}"

        cv2.putText(frame, label,
                    (tx, ty - 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(frame, conf_label,
                    (tx, ty - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 1)

    cv2.imshow("Age & Gender Detection", frame)
    if (cv2.waitKey(1) & 0xFF) == ord('x'):
        break

cap.release()
cv2.destroyAllWindows()